# System 1: Interactive Model Selection & Hardware Benchmark Studio (AIC 2026)
### Trình Đánh Giá So Sánh Mô Hình Đa Phương Thức, Từ Điển Bản Địa & Tối Ưu Hóa GPU/TPU Trên Kaggle
- **Mục tiêu:** Cung cấp môi trường kiểm thử trực quan, đo lường độ trễ (latency), Recall@K và Cosine Margin giữa các mô hình:
  1. **Vision Embedding:** `SigLIP Base` (768d) vs `SigLIP SO400M` (1152d) vs `ViSigLIP-OT` (768d) vs `Jina-CLIP` (768d).
  2. **OCR 2-Tier Engine:** `EasyOCR` / `PaddleOCR` (Tier 1 Fast Path) vs `Vintern-1B-v3_5` (Tier 2 SOTA VLM).
  3. **ASR Voice-to-Text:** `faster-whisper-large-v3 (INT8)` vs `NVIDIA FastConformer`.
  4. **Query Mode Simulator:** Bật/tắt linh hoạt 3 chế độ tìm kiếm (Dual-Consensus, Fast-Vi, Fast-En).
- **Hỗ trợ phần cứng:** Tự động phát hiện Kaggle Dual GPU T4, Kaggle TPU VM v3-8 (PyTorch-XLA), hoặc Local CPU.

In [ ]:
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
!pip install -q faster-whisper easyocr faiss-cpu transformers accelerate open_clip_torch Pillow numpy pandas tqdm pyyaml

In [ ]:
# 2. TỰ ĐỘNG PHÁT HIỆN VÀ KHỞI TẠO PHẦN CỨNG (GPU / TPU / CPU)
import torch
import os
import sys

device_info = {"type": "cpu", "count": 1, "devices": ["cpu"]}

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    tpu_dev = xm.xla_device()
    device_info = {"type": "tpu", "count": 8, "devices": [tpu_dev]}
    print("Da phat hien Kaggle TPU VM v3-8! Khoi tao engine PyTorch-XLA thanh cong.")
except Exception:
    if torch.cuda.is_available():
        n_gpu = torch.cuda.device_count()
        devices = [f"cuda:{i}" for i in range(n_gpu)]
        device_info = {"type": "cuda", "count": n_gpu, "devices": devices}
        print(f"Da phat hien {n_gpu} NVIDIA GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpu)]}")
    else:
        print("Chay tren CPU mode (Local / Lightweight Benchmark).")

print(f"Device Strategy: {device_info}")

In [ ]:
# 3. BỘ TỪ ĐIỂN KHÁI NIỆM BẢN ĐỊA & MODULE LÀM GIÀU TRUY VẤN TRUNG THỰC (FAITHFUL ENRICHER)
import re
import unicodedata

VIETNAMESE_CULTURAL_LEXICON = {
    "múa lân": {
        "canonical_name": "Múa lân",
        "aliases": ["múa lân", "con lân", "đầu lân", "lân sư rồng", "múa sư tử"],
        "visual_anchor_en": "traditional Vietnamese lion dance costume performance with lion head and drums",
        "keywords_vi": ["múa lân", "lân sư rồng", "đầu lân"]
    },
    "áo dài": {
        "canonical_name": "Áo dài",
        "aliases": ["áo dài", "áo dài truyền thống", "áo dài cách tân"],
        "visual_anchor_en": "Vietnamese traditional long tunic dress worn over trousers",
        "keywords_vi": ["áo dài", "áo dài truyền thống"]
    },
    "nón lá": {
        "canonical_name": "Nón lá",
        "aliases": ["nón lá", "nón bài thơ", "nón chóp lá cọ"],
        "visual_anchor_en": "traditional conical palm leaf hat worn by Vietnamese people",
        "keywords_vi": ["nón lá", "nón bài thơ"]
    },
    "chợ nổi": {
        "canonical_name": "Chợ nổi",
        "aliases": ["chợ nổi", "chợ trên sông", "chợ nổi cái răng"],
        "visual_anchor_en": "vibrant floating market with wooden boats and canoes selling fruits on river canal",
        "keywords_vi": ["chợ nổi", "trên sông"]
    },
    "bánh chưng": {
        "canonical_name": "Bánh chưng",
        "aliases": ["bánh chưng", "bánh tét", "bánh chưng xanh"],
        "visual_anchor_en": "square green sticky rice cake wrapped in banana or dong leaves tied with bamboo strings",
        "keywords_vi": ["bánh chưng", "bánh tét"]
    }
}

def remove_accents(text):
    if not text: return ""
    text = text.replace("đ", "d").replace("Đ", "D")
    normalized = unicodedata.normalize("NFD", text)
    return "".join(c for c in normalized if unicodedata.category(c) != "Mn")

def lookup_cultural_concepts(query_vi):
    if not query_vi: return []
    q_lower = query_vi.lower().strip()
    q_no_acc = remove_accents(q_lower)
    detected = []
    matched_keys = set()
    for key, entity in VIETNAMESE_CULTURAL_LEXICON.items():
        if key in matched_keys: continue
        for alias in entity["aliases"]:
            al_low = alias.lower()
            al_no = remove_accents(al_low)
            m1 = re.search(rf"\b{re.escape(al_low)}\b", q_lower)
            m2 = re.search(rf"\b{re.escape(al_no)}\b", q_no_acc)
            if m1 or m2:
                sp = m1.start() if m1 else m2.start()
                detected.append({"start_pos": sp, "canonical_name": entity["canonical_name"], "visual_anchor_en": entity["visual_anchor_en"], "keywords_vi": entity["keywords_vi"]})
                matched_keys.add(key)
                break
    detected.sort(key=lambda x: x["start_pos"])
    return detected

def enrich_query_faithfully(query_vi, translated_en=""):
    concepts = lookup_cultural_concepts(query_vi)
    base_en = translated_en.strip() if translated_en else query_vi.strip()
    if concepts:
        anchors = " | ".join([c["visual_anchor_en"] for c in concepts])
        enriched_en = f"{base_en} ({anchors})"
    else:
        enriched_en = base_en
    return {"raw_vi": query_vi, "enriched_en": enriched_en, "concepts": [c["canonical_name"] for c in concepts]}

In [ ]:
# 4. TEST LÀM GIÀU TRUY VẤN VẬT THỂ THUẦN VIỆT
sample_queries = [
    ("Người múa lân trên đường phố", "People performing lion dance on the street"),
    ("Cô gái mặc áo dài xanh đội nón lá", "Woman wearing blue long dress and conical hat"),
    ("Chợ nổi trên sông có nhiều ghe thuyền", "Floating market on the river with many boats"),
    ("Cầu thủ sút bóng trên sân cỏ", "Football player shooting on the pitch")
]

print("=" * 75)
print("KẾT QUẢ LÀM GIÀU TRUY VẤN TRUNG THỰC (NO HALLUCINATION):")
print("=" * 75)
for q_vi, q_en in sample_queries:
    res = enrich_query_faithfully(q_vi, q_en)
    print(f"* Goc (VI)  : {res['raw_vi']}")
    print(f"  Thuc the  : {res['concepts']}")
    print(f"  Enriched  : {res['enriched_en']}")
    print("-" * 75)

In [ ]:
# 5. TRÌNH MÔ PHỎNG CHẾ ĐỘ TRUY VẤN ON/OFF (QUERY MODE TOGGLE SIMULATOR)
class QueryModeSimulator:
    def __init__(self):
        self.active_mode = "mode1_dual_consensus" # mode1_dual_consensus, mode2_fast_vi, mode3_fast_en
        
    def set_mode(self, mode_name):
        assert mode_name in ["mode1_dual_consensus", "mode2_fast_vi", "mode3_fast_en"], "Mode khong hop le"
        self.active_mode = mode_name
        print(f"[Mode Switched] -> Chế độ hiện tại: {mode_name}")
        
    def route_query(self, query_vi, mock_translate_api=None):
        import time
        t0 = time.perf_counter()
        
        if self.active_mode == "mode2_fast_vi":
            # Mode 2: Chi dung Tieng Viet Native (ViSigLIP) khong qua API dich
            routed_info = {"mode": "Mode 2: Fast Vi-Only", "target_model": "ViSigLIP-OT (768d)", "query_used": query_vi, "use_translation": False}
        elif self.active_mode == "mode3_fast_en":
            # Mode 3: Dich sang Tieng Anh + Lam giau (SigLIP SO400M / Base)
            trans_en = mock_translate_api(query_vi) if mock_translate_api else query_vi
            en_res = enrich_query_faithfully(query_vi, trans_en)
            routed_info = {"mode": "Mode 3: Fast En-Only", "target_model": "SigLIP SO400M (1152d)", "query_used": en_res["enriched_query_en"], "use_translation": True}
        else:
            # Mode 1: Dual Stream Consensus (Vi-Query cho ViSigLIP va En-Query cho SigLIP SO400M)
            trans_en = mock_translate_api(query_vi) if mock_translate_api else query_vi
            en_res = enrich_query_faithfully(query_vi, trans_en)
            routed_info = {"mode": "Mode 1: Dual-Stream Consensus", "stream_a": {"model": "ViSigLIP-OT (768d)", "query": query_vi}, "stream_b": {"model": "SigLIP SO400M (1152d)", "query": en_res["enriched_query_en"]}, "fusion": "RRF (Reciprocal Rank Fusion)"}
            
        t_elap_ms = (time.perf_counter() - t0) * 1000.0
        routed_info["routing_latency_ms"] = round(t_elap_ms, 3)
        return routed_info

sim = QueryModeSimulator()
def mock_translate(q): return "People performing lion dance" if "múa lân" in q.lower() else "Woman wearing long dress"

print("\n--- TEST 3 CHẾ ĐỘ TRUY VẤN ON/OFF ---")
sim.set_mode("mode1_dual_consensus")
print(sim.route_query("Người múa lân", mock_translate))

sim.set_mode("mode2_fast_vi")
print(sim.route_query("Người múa lân", mock_translate))

sim.set_mode("mode3_fast_en")
print(sim.route_query("Người múa lân", mock_translate))

In [ ]:
# 6. TỔNG KẾT & HƯỚNG DẪN CHẠY TRÊN KAGGLE
print("=" * 75)
print("HƯỚNG DẪN TRIỂN KHAI PHÂN ĐOẠN 4-NOTEBOOK TRÊN KAGGLE:")
print("=" * 75)
print("1. Notebook 01: Bóc tách Keyframe + HSV Hist + Laplacian + WebP Thumbnails.")
print("2. Notebook 02: Bóc tách ASR (faster-whisper INT8) + OCR 2-Tier (EasyOCR/Vintern-1B).")
print("3. Notebook 03: Trích xuất ma trận Vector Dual SigLIP SO400M (1152d) & ViSigLIP (768d).")
print("4. Notebook 04: Master Assembler hợp nhất Timeline, khử trùng lặp ảo và đóng gói release_artifacts.zip.")
print("=" * 75)